# 10 · SkewerJAX and fake_spectra: speed and memory

How long SkewerJAX and fake_spectra take to compute the same spectra, where the time goes, and how much memory they
use, on three TNG50 snapshots of growing size. What the two codes compute, and how close they are, is in notebook
`09_comparison_fake_spectra_results`.

**What is compared.** The same skewers (`benchmark_skewers(N)`: nested sets, the first $N$ of a fixed list along the
three axes), the same pixel width (1 km/s), the same two lines ($\mathrm{Ly}\alpha$ and Si III 1206), the same snapshot.
fake_spectra runs its own pipeline (`Spectra`, then `get_tau`). SkewerJAX runs two modes in the same process, on one shared
cell selection:
- **matched**: fake_spectra's scheme on fake_spectra's pixel grid (`FAKE_SPECTRA_EXTRACTION` +
  `FAKE_SPECTRA_INTEGRATION`, `matched_tau`), the like-for-like comparison;
- **default**: SkewerJAX's own choices (Voronoi pieces of 0.25 km/s dual segments, exact frame,
  `velocity_period=True`), what a user runs.

On TNG50-4 a third mode, **float32** (the default in single precision), shows what the precision costs.

**Protocol** (`skewerjax.comparison.benchmark`). Each configuration (code, snapshot, $N$, device, threads) runs in its
own process. The first run includes the JIT compilation and a cold read; it is kept apart, and the compilation is
reported separately (first run minus the median timed run). The timed runs give the median with the min–max spread.
Every JAX result is pulled to the host, which waits until it is ready. The wall time is split into exclusive stages:
**setup**, **I/O** (raw HDF5 reads), **cell selection** (the cells near the skewers), **gas physics** (ion fractions,
temperatures), **pieces** (the deposition of the cells on the skewers) and the **$\tau$ sum**. fake_spectra deposits
its particles inside its C engine, so its pieces are in its $\tau$.

**Two presets** (`--preset` of `scripts/benchmarks/benchmark_fake_spectra.py`, `PRESET` below):
- **full**: $N = 1, 4, \ldots, 1024$ skewers on each snapshot, and 1 to 16 threads on TNG50-4 at $N = 16$; caps of 1 h
  and 48 GB per point, larger $N$ skipped when the two previous points extrapolate beyond them; ten hours or more.
  It was started and stopped after 3.3 h: it measured the whole TNG50-4 series and TNG50-3 up to $N = 64$.
- **quick**: a reduced study in at most 30 min of new runs, with a cap and an estimate per point: TNG50-3 at
  $N = 1, 4, 16$ (two points re-measured, the others kept from the full run), TNG50-2 at $N = 1$, and 1 to 16 threads
  on TNG50-4 at $N = 1$ (one or two timed runs). It took 26.5 min.

The results file merges both; every record keeps its preset, printed below and in the appendix. The points that
did not run are listed below and missing from the figures.

The path: the setup; the time against $N$; where the time goes; the speed-up; the time against the snapshot size; the
memory; the threads; the agreement of the benchmark spectra; a summary of what these points can and cannot show.

Runtime: ~15 s from the saved results; the quick benchmark takes ~30 min on this machine (`RUN_BENCHMARKS = True`, or
`python scripts/benchmarks/benchmark_fake_spectra.py --preset quick`).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")          # this kernel only reads and plots; the benchmark workers set their own
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import json, time
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, add_panel_letters, plot_bench_vs, plot_bench_speedup, plot_bench_stages,
                                      plot_bench_accuracy, BENCH_STAGE_COLORS, BENCH_STAGE_LABELS)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes
from skewerjax.comparison import benchmark as B
from skewerjax.utils.paths import DATA_DIR, BENCHMARK_RAW_DIR, BENCHMARK_RECORDS_DIR


def legend_below(fig, axs, ncol=4):
    """One legend for the whole figure, below the panels: every series once."""
    h = {l: a for ax in np.ravel(axs) for a, l in zip(*ax.get_legend_handles_labels())}
    fig.legend(h.values(), h.keys(), loc="outside lower center", ncol=ncol, fontsize=15, frameon=False)

## The setup

`RUN_BENCHMARKS = False` reads the saved results (`BENCHMARK_RECORDS_DIR/fake_spectra_benchmark.json`). `True`
measures every configuration of `PRESET` that has no record yet in `BENCHMARK_RAW_DIR/fake_spectra` (a run resumes;
delete a record to measure it again; the full preset re-measures the quick points that hold fewer timed runs), then
collects them; `BENCH_SETS = "smoke"` is a 4-minute check on TNG50-4 with 1 and 4 skewers, written to separate files.
The machine, the libraries, the thread settings, the snapshots and which preset measured which point are printed below.

**Threads.** Each worker is pinned to its CPU set (`os.sched_setaffinity`). OpenMP, OpenBLAS and MKL get the thread
count of the configuration; fake_spectra runs its OpenMP loops on it; XLA uses the CPUs of the set (one Eigen thread
with one thread). `XLA_PYTHON_CLIENT_PREALLOCATE=false`, so the GPU memory is what XLA uses, not a preallocated pool.
The GPU runs keep the same host threads. **The machine was shared**: another user's 16-process job ran through both
launches, so the timings carry a few tens of per cent of noise (the min–max spread of the timed runs shows it).

In [ ]:
RUN_BENCHMARKS = False            # True: run every configuration of PRESET that has no record yet (resumes), then plot
PRESET = "quick"                  # "quick": the reduced study (<= 30 min of new runs); "full": the whole study (~10 h or more)
BENCH_SETS = "scaling,threads"    # or "smoke": TNG50-4, 1 and 4 skewers (~4 min), written to separate files
tag = "_smoke" if BENCH_SETS == "smoke" else ""
RAW = BENCHMARK_RAW_DIR / f"fake_spectra{tag}"                         # one record per configuration ($SKEWERJAX_BENCHMARK_RAW)
RESULTS = BENCHMARK_RECORDS_DIR / f"fake_spectra_benchmark{tag}.json"  # all of them, what this notebook reads ($SKEWERJAX_BENCHMARK_RESULTS)
if RUN_BENCHMARKS:                # the same as: python scripts/benchmarks/benchmark_fake_spectra.py --preset <PRESET> --sets <BENCH_SETS>
    t0 = time.perf_counter()
    B.run_suite([c for s in BENCH_SETS.split(",") for c in B.benchmark_sets(s, DATA_DIR, preset=PRESET)], RAW,
                total_budget_s=B.QUICK_BUDGET_S if PRESET == "quick" else None, log=print)
    wall = time.perf_counter() - t0
    B.collect(RAW, RESULTS, sets=BENCH_SETS.split(","), launcher_wall_s=wall,
              launches=[dict(preset=PRESET, sets=BENCH_SETS.split(","), wall_s=wall, date=time.strftime("%Y-%m-%d %H:%M"))])
res = json.loads(RESULTS.read_text())
rows = B.result_rows(res["records"])
SIMS = [s["sim"] for s in res["snapshots"]]
CELLS = {s["sim"]: s["n_gas"] for s in res["snapshots"]}
B.print_setup(res)
by = {}                                                   # which preset measured which point
for r in B.select_rows(rows):
    by.setdefault((r["preset"], r["sim"], r["threads"] == 32), set()).add(r["n_skewers"] if r["threads"] == 32 else r["threads"])
for (p, sim, scaling), pts in sorted(by.items(), key=lambda x: (x[0][0], CELLS[x[0][1]], not x[0][2])):
    print(f"{p:5s} preset, {sim}: " + (f"N = {sorted(pts)} skewers (32 threads)" if scaling else f"threads {sorted(pts)}"))

def not_measured(ax, sim):
    """A note on a panel of a snapshot where SkewerJAX has no measured point (left for the full preset)."""
    if not B.select_rows(rows, sim=sim, code="skewerjax", threads=32):
        ax.text(0.5, 0.5, "SkewerJAX not measured here:\nover the quick cap\n(left for the full preset)", transform=ax.transAxes,
                ha="center", va="center", fontsize=15, color="0.35")

## The time against the number of skewers

The wall time of one run (top, log–log, median with the min–max spread; the slope is fitted for $N \geq 16$) and the
throughput in skewers per second (bottom), on each snapshot. Colour: the code and the device; line style: the
SkewerJAX mode (solid default, dashed matched, dotted float32); open markers: the modes other than the default.

In [ ]:
fig, axs = plt.subplots(2, len(SIMS), figsize=(22, 15), layout="constrained", sharex=True, squeeze=False)
slopes = {}
for j, sim in enumerate(SIMS):
    slopes[sim] = plot_bench_vs(axs[0, j], rows, sim=sim, threads=32, fit_from=16)
    plot_bench_vs(axs[1, j], rows, sim=sim, threads=32, what="throughput")
    axs[0, j].set_title(f"{sim}: {CELLS[sim] / 1e6:,.0f} M gas cells")
    axs[0, j].legend(loc="upper center", bbox_to_anchor=(0.5, -0.04), fontsize=15, frameon=False)
    axs[0, j].set_xlabel("")
    not_measured(axs[0, j], sim); not_measured(axs[1, j], sim)
add_panel_letters(axs)
fig.suptitle("SkewerJAX's time is linear in the number of skewers; fake_spectra's is set by the snapshot size", fontsize=21); plt.show()
for sim, s in slopes.items():
    print(sim, "log-log slope of the time per run for N >= 16:", {f"{k[0]} {k[1]} {k[2] or ''}".strip(): None if v is None else round(v, 2) for k, v in s.items()})

*Takeaway.* SkewerJAX's time grows almost linearly with $N$ (slope 0.94 on TNG50-4, CPU default): about 1.8 s per
skewer at 32 threads, nearly all of it the $\tau$ sum. fake_spectra grows far less (slope 0.41 on TNG50-4, 0.15 on
TNG50-3), because its cost is dominated by passes over the whole snapshot that do not depend on $N$. float32 halves
SkewerJAX's time at large $N$; on this consumer GPU every mode is slower than on the CPU. On TNG50-2 only fake_spectra
was measured: SkewerJAX needed more than the quick cap (more than 600 s for the first run and one timed run).

## Where the time goes

The median wall time of one run split into its stages: few and many skewers on the small snapshot, and one skewer on
the large one. The JIT compilation of the first run is written after each bar; it is not in the bars.

In [ ]:
POINTS = [("TNG50-4", 1), ("TNG50-4", 64), ("TNG50-2", 1)]            # (simulation, N): few and many skewers, small and large snapshot
POINTS = [(s, n) for s, n in POINTS if B.select_rows(rows, sim=s, n_skewers=n, threads=32)]
fig, axs = plt.subplots(len(POINTS), 1, figsize=(22, 5.2 * len(POINTS)), layout="constrained", squeeze=False)
for k, (ax, (sim, n)) in enumerate(zip(axs[:, 0], POINTS)):
    plot_bench_stages(ax, rows, sim=sim, n_skewers=n, threads=32)
    ax.set_title(f"({'abc'[k]}) {sim}, N = {n}", loc="left")
    ax.set_xlim(right=ax.get_xlim()[1] * 1.35)                         # room for the totals
axs[0, 0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.12), ncol=6, frameon=False, fontsize=15)
fig.suptitle("SkewerJAX's time goes to the τ sum; fake_spectra's to reading and processing the whole snapshot", fontsize=21); plt.show()

*Takeaway.* SkewerJAX spends its time in the $\tau$ sum: an exact Voigt profile for every piece at every pixel
(103 of 117 s at $N = 64$, default). Its selection and I/O are small on TNG50-4 (1–9 s). Its default kernel also
compiles once per process (about 17 s on the CPU, 6 s on the GPU), which matters only for a few skewers. fake_spectra
at $N = 1$ is mostly I/O and gas physics. On TNG50-2 these are 90 s and 58 s of its 151 s per run: it reads and
processes every particle of the 374 GB snapshot whatever $N$.

## The speed-up over fake_spectra

The time of fake_spectra divided by the time of each SkewerJAX series at the same point (above 1: SkewerJAX faster;
below 1: slower). The table gives the default mode's times.

In [ ]:
fig, axs = plt.subplots(1, len(SIMS), figsize=(22, 8.5), layout="constrained", sharey=True, squeeze=False)
for ax, sim in zip(axs[0], SIMS):
    plot_bench_speedup(ax, rows, sim=sim, threads=32)
    ax.set_title(sim)
    not_measured(ax, sim)
legend_below(fig, axs, ncol=3)
add_panel_letters(axs[0])
fig.suptitle("fake_spectra is faster except for a few skewers on the larger snapshot", fontsize=21); plt.show()
for sim in SIMS:
    for n in sorted({r["n_skewers"] for r in B.select_rows(rows, sim=sim, threads=32)}):
        cpu, gpu = (B.value(rows, sim=sim, n_skewers=n, threads=32, code="skewerjax", platform=p, mode="default") for p in ("cpu", "gpu"))
        fs = B.value(rows, sim=sim, n_skewers=n, threads=32, code="fake_spectra")
        t = lambda v, ref: "    (not run)       " if np.isnan(v) else f"{v:8.3g} s ({v / ref:5.1f}× its time)"
        print(f"{sim} N={n:5d}: fake_spectra {fs:7.3g} s | SkewerJAX default, CPU {t(cpu, fs)}, GPU {t(gpu, fs)}"
              + ("" if np.isnan(cpu * gpu) else f" | GPU / CPU {gpu / cpu:.2f}"))

*Takeaway.* On TNG50-4 fake_spectra is faster everywhere: 1.7 times at $N = 1$ and 65 times at $N = 256$ (CPU default). On
TNG50-3 the fixed cost of fake_spectra's whole-snapshot passes (25–32 s per run) exceeds SkewerJAX's cost at $N = 1$
(15 s), so SkewerJAX is faster there; the two cross between $N = 1$ and $N = 4$. The crossover moves to larger $N$ as the
snapshot grows only if SkewerJAX's own selection stays cheaper than fake_spectra's reading, which TNG50-2 has not
tested yet.

## The time against the snapshot size

At $N = 1$ skewer, the one count measured on the three snapshots (19, 152 and 1205 million gas cells): the total, the
I/O and selection (reading and choosing the cells) and the computation (physics, pieces, $\tau$). The slopes are
log–log fits: on three points for fake_spectra, on two for SkewerJAX (TNG50-2 not measured).

In [ ]:
N_SIZE = 1                                      # the one N measured on the three snapshots (the quick preset)
fig, axs = plt.subplots(1, 3, figsize=(22, 8.5), layout="constrained", squeeze=False)
size_slopes = {}
for ax, what in zip(axs[0], ("total", "io+selection", "compute")):
    size_slopes[what] = plot_bench_vs(ax, rows, x="n_gas", what=what, n_skewers=N_SIZE, threads=32, fit_from=0)
    ax.set_xticks(list(CELLS.values()), [f"{s}\n{c / 1e6:,.0f} M" for s, c in CELLS.items()], fontsize=15); ax.minorticks_off()
for ax in axs[0]:
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=15, frameon=False)
add_panel_letters(axs[0])
fig.suptitle(f"At N = {N_SIZE}, both codes scale with the snapshot: fake_spectra entirely, SkewerJAX through its selection", fontsize=21); plt.show()
for what, s in size_slopes.items():
    print(f"slope against the number of cells, {what:12s}:", {f"{k[0]} {k[1]} {k[2] or ''}".strip(): None if v is None else round(v, 2) for k, v in s.items()})

*Takeaway.* At fixed $N$ both codes scale with the snapshot. fake_spectra scales entirely (slope 1.1 for the total):
it reads and processes every particle. SkewerJAX's I/O and selection scale the same way (slope 1.1, two points),
since its streamed selection also passes over every cell. Its computation hardly grows (slope 0.3), because it only
sees the cells near the skewers. With two points per SkewerJAX series these slopes are indications, not fits.

## The memory

The peak host memory (RSS) and the peak GPU memory against $N$ on each snapshot (columns), and against the snapshot
size at $N = 1$ (right). The host peak of a SkewerJAX mode is sampled during that mode (from the selected cells
onwards); the GPU peak is XLA's `peak_bytes_in_use`, cumulative over the modes of the process (matched, then default,
then float32).

In [ ]:
fig, axs = plt.subplots(2, len(SIMS) + 1, figsize=(22, 12.5), layout="constrained", squeeze=False)
for i, what in enumerate(("rss", "gpu")):
    for j, sim in enumerate(SIMS):
        plot_bench_vs(axs[i, j], rows, sim=sim, threads=32, what=what)
        axs[i, j].set_title(sim if i == 0 else "")
        not_measured(axs[i, j], sim)
        if what == "gpu":
            axs[i, j].set(yscale="linear", ylim=(0, None))
    plot_bench_vs(axs[i, -1], rows, x="n_gas", what=what, n_skewers=N_SIZE, threads=32)
    axs[i, -1].set_xticks(list(CELLS.values()), [s.replace("TNG50-", "-") for s in CELLS], fontsize=15); axs[i, -1].minorticks_off()
    axs[i, -1].set(ylabel="", xlabel=f"snapshot (N = {N_SIZE})")
    if what == "gpu":
        axs[i, -1].set(yscale="linear", ylim=(0, None))
    axs[0, -1].set_title("simulation size")
legend_below(fig, axs, ncol=4)
add_panel_letters(axs)
fig.suptitle("SkewerJAX's host memory grows with the number of skewers; fake_spectra's stays near 1 GB", fontsize=21); plt.show()

*Takeaway.* fake_spectra needs about 0.4 GB on TNG50-4 and about 1 GB on the larger snapshots, whatever $N$.
SkewerJAX needs 1.3–2 GB for one skewer and grows by about 80 MB per skewer in the default mode (20.6 GB at
$N = 256$ on TNG50-4); the matched mode grows less (4.7 GB). The GPU holds at most 1 GB.

## The threads

TNG50-4, $N = 1$ in the quick preset ($N = 16$ in the full one): the time against the number of CPU threads (left),
and the stages with one core for every code and mode (right; the strict single-core comparison). The one-core run
also measures `crossings`, the mode that integrates each absorber over its exact crossing of the skewer.

In [ ]:
N_THREADS = max(r["n_skewers"] for r in B.select_rows(rows, sim="TNG50-4", threads=1))   # 1 (quick) or 16 (full)
fig, axs = plt.subplots(1, 2, figsize=(22, 8), layout="constrained", gridspec_kw={"width_ratios": [1, 1.5]})
plot_bench_vs(axs[0], rows, x="threads", sim="TNG50-4", n_skewers=N_THREADS, fit_from=1, modes=("matched", "default"))
axs[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=15, frameon=False); axs[0].set_title(f"TNG50-4, N = {N_THREADS}")
plot_bench_stages(axs[1], rows, sim="TNG50-4", n_skewers=N_THREADS, threads=1)
axs[1].set_xscale("log"); axs[1].set_xlim(0.5, axs[1].get_xlim()[1] * 8); axs[1].set_title("one core each (stacked, log axis)", loc="left")
axs[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=15, frameon=False)
add_panel_letters(axs)
fig.suptitle(f"Threads help SkewerJAX (about 6× from 1 to 32), not fake_spectra at N = {N_THREADS}", fontsize=21); plt.show()
B.print_rows(rows, sim="TNG50-4", n_skewers=N_THREADS, threads=1)

*Takeaway.* From 1 to 32 threads SkewerJAX's default run goes from 18 s to 3.0 s (about 6 times). fake_spectra stays at
2.4 s, because at $N = 1$ it waits for its read, not its OpenMP loops. On one core fake_spectra is 7.5 times faster than
SkewerJAX's default; float32 and the crossings mode halve and third the gap. These are single runs on a shared machine
at one skewer: the dip at 8 threads and the rise at 16 are within the noise.

## The agreement of the benchmark spectra

The benchmark also checks the spectra it times. Matched mode against fake_spectra: the max $\vert\Delta F\vert$ over the
first $\min(N, 64)$ skewers and the rms of $\Delta F$ over their pixels; default mode: the difference in the mean flux
over all skewers (a real difference of the two schemes, explained in notebook 09).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(22, 8), layout="constrained", sharey=True)
for ax, line, title in zip(axs, ("HI-LyA", "SiIII-1206"), (r"Ly$\alpha$", "Si III 1206")):
    plot_bench_accuracy(ax, rows, line=line, sims=SIMS); ax.set_title(title)
axs[1].set_ylabel("")
legend_below(fig, axs, ncol=3)
add_panel_letters(axs)
fig.suptitle("While timed, the matched mode reproduces fake_spectra: rms ΔF ≤ 5e-6", fontsize=21); plt.show()
for r in B.select_rows(rows, code="skewerjax", platform="cpu", mode="matched", threads=32):
    a = r["agreement"] or {}
    print(f"{r['sim']} N={r['n_skewers']:5d}: " + " | ".join(f"{l}: max|dF| {a[l]['max_dF']:.1e}, rms {a[l]['rms_dF']:.1e} on {a[l]['n_skewers']} skewers, "
                                                             f"<F> fake_spectra {a[l]['meanF_fake_spectra']:.4f} default {a[l]['meanF_default']:.4f}"
                                                             for l in ("HI-LyA", "SiIII-1206") if l in a))

*Takeaway.* The matched mode reproduces fake_spectra: rms $\Delta F \leq 5\times10^{-6}$ for $\mathrm{Ly}\alpha$ and
Si III on every point. The max $\vert\Delta F\vert$ rises from $2\times10^{-5}$ at $N \leq 4$ to $2.2\times10^{-4}$ at
$N \geq 16$, set by one pixel of one skewer per line, which enters the stored set at $N = 16$. The default mode's mean
flux differs by about 0.03 in $\mathrm{Ly}\alpha$: the effects of notebook 09, mostly the 7/8 normalisation.

## Summary

- **The time against $N$.** SkewerJAX is linear in the number of skewers (about 1.8 s per skewer on 32 CPU threads,
  mostly the $\tau$ sum). fake_spectra's cost is dominated by passes over the whole snapshot: nearly flat in $N$, and
  linear in the snapshot size.
- **The speed-up.** On TNG50-4 fake_spectra is 1.7 times (one skewer) to 65 times (256 skewers) faster. On TNG50-3 SkewerJAX
  is faster for one skewer and slower from about four.
- **The levers.** float32 halves SkewerJAX's time. The consumer GPU does not help. Threads give about 6 times from 1 to 32.
- **Memory.** About 80 MB per skewer for SkewerJAX's default, flat ~1 GB for fake_spectra.
- **Agreement.** The matched mode agrees with fake_spectra to rms $5\times10^{-6}$ while it is timed.
- **What the quick study cannot show.**
  - TNG50-2 has only fake_spectra (one skewer); SkewerJAX's TNG50-2 points need more than the quick cap and are left
    for the full preset.
  - The snapshot-size slopes of SkewerJAX rest on two points.
  - The threads scan is one skewer, one timed run each.
  - Every point shares the machine with another user's job.
  - Scaling up (full preset): TNG50-2 at $N = 1, 4, 16$, TNG50-3 on the GPU at $N = 64$, the threads at $N = 16$.

## Appendix: all the measurements

One row per configuration and mode: the stage medians [s], the total and its spread over the timed runs, the first
run and the compilation [s], the peak host and GPU memory [GB], and the preset that measured it.

In [ ]:
B.print_rows(rows)